# Laboratorio Dirigido N.° 05
## Motor de decisión ante riesgo cambiario

**Curso:** Analítica Empresarial Integrada  
**Estudiante:** Cuya Vera Juandiego Alejandro  
**Semana:** 5  
**Caso:** una empresa importadora peruana debe pagar **USD 100 000 dentro de 60 días**.  
**Fuente:** API pública de Series Estadísticas del Banco Central de Reserva del Perú.

### Pregunta orientadora
¿Cómo tomar una decisión defendible cuando no conocemos con certeza el tipo de cambio futuro?

> Este notebook se desarrolla íntegramente durante la clase con acompañamiento docente. No constituye una tarea, exposición ni entregable.

## Ruta de trabajo

1. Conexión y parámetros del caso.  
2. Descarga y trazabilidad de las series del BCRP.  
3. Limpieza y control de calidad.  
4. Selección explícita de la serie de venta.  
5. Retornos y volatilidad.  
6. Simulación Monte Carlo.  
7. Comparación entre comprar hoy y esperar.  
8. Motor transparente de decisión.  
9. Sensibilidad y reto en clase.

## BLOQUE 1 — Conexión, trazabilidad y calidad

### Celda 1 — Entorno de trabajo

In [ ]:
# Librerías disponibles por defecto en Google Colab.
import sys
import platform
from datetime import date, timedelta

import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)

print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
print("Entorno listo.")

Python: 3.13.15
Plataforma: Linux-6.6.122+-x86_64-with-glibc2.39
Entorno listo.


### Celda 2 — Parámetros visibles y editables

In [ ]:
# Parámetros empresariales
MONTO_USD = 100_000  # Define el monto que la empresa debe pagar en dólares
HORIZONTE_CALENDARIO = 60  # Define que el pago se realizará dentro de 60 días
ESCENARIOS = 20_000  # Define la cantidad de escenarios que se simularán
VENTANA_BASE = 252  # Define las observaciones diarias usadas como base histórica
SEMILLA = 2026  # Fija la semilla para obtener resultados reproducibles

# Umbrales académicos del motor de decisión
TOLERANCIA_PROB_SOBRECOSTO = 0.40  # Establece 40 % como límite de probabilidad de sobrecosto
TOLERANCIA_SOBRECOSTO_P95 = 0.02  # Establece 2 % como límite de sobrecosto en el P95

# Series diarias del BCRP: compra y venta interbancaria (S/ por US$)
SERIES = ["PD04637PD", "PD04638PD"]  # Guarda los códigos de las dos series del tipo de cambio del BCRP
FECHA_FIN = date.today()  # Obtiene la fecha actual como fecha final
FECHA_INICIO = FECHA_FIN - timedelta(days=5 * 365)  # Calcula una fecha inicial de aproximadamente 5 años antes

print(f"Obligación: USD {MONTO_USD:,.0f}")  # Muestra el monto de la obligación en dólares
print(f"Horizonte: {HORIZONTE_CALENDARIO} días calendario")  # Muestra el plazo del pago
print(f"Escenarios: {ESCENARIOS:,}")  # Muestra la cantidad de escenarios a simular
print(f"Periodo solicitado: {FECHA_INICIO} a {FECHA_FIN}")  # Muestra el periodo de datos que se solicitará

Obligación: USD 100,000
Horizonte: 60 días calendario
Escenarios: 20,000
Periodo solicitado: 2021-09-18 a 2026-09-17


### Celda 3 — Consulta a la API del BCRP

Se solicitan las dos series por su código oficial. La respuesta se valida antes de continuar.

In [ ]:
BASE_API = "https://estadisticas.bcrp.gob.pe/estadisticas/series/api"
# Guarda la dirección base de la API de estadísticas del BCRP

codigos = "-".join(SERIES)
# Une los códigos de las series con "-" para formar parte de la URL

URL = f"{BASE_API}/{codigos}/json/{FECHA_INICIO.isoformat()}/{FECHA_FIN.isoformat()}/esp"
# Construye la URL completa para solicitar las series en formato JSON y en español

respuesta = requests.get(URL, timeout=60)
# Envía una solicitud al BCRP y establece un máximo de 60 segundos para recibir respuesta

print("Estado HTTP:", respuesta.status_code)
# Muestra el código HTTP para saber si la solicitud fue procesada correctamente

respuesta.raise_for_status()
# Genera un error si la respuesta HTTP indica que hubo un problema

datos_json = respuesta.json()
# Convierte la respuesta del BCRP de JSON a un objeto de Python

if "config" not in datos_json or "periods" not in datos_json:
    raise ValueError("La respuesta del BCRP no contiene 'config' y 'periods'.")
# Verifica que la respuesta tenga las secciones necesarias: configuración y periodos

print("Series recibidas:", len(datos_json["config"]["series"]))
# Cuenta y muestra cuántas series fueron recibidas del BCRP

print("Periodos recibidos:", len(datos_json["periods"]))
# Cuenta y muestra cuántos periodos de datos fueron recibidos

Estado HTTP: 200
Series recibidas: 2
Periodos recibidos: 1304


### Celda 4 — Trazabilidad: nombres y códigos de las series

In [ ]:
config_series = datos_json["config"]["series"]
# Extrae de la respuesta del BCRP la información de las series recibidas

trazabilidad = pd.DataFrame([
    {
        "codigo": s.get("code", SERIES[i] if i < len(SERIES) else ""),
        # Obtiene el código de la serie; si no existe, utiliza el código definido previamente

        "nombre": s.get("name", ""),
        # Obtiene el nombre oficial de la serie

        "unidad": s.get("unit", ""),
        # Obtiene la unidad en la que se expresa la serie
    }
    for i, s in enumerate(config_series)
])
# Construye un DataFrame con la información de cada serie

display(trazabilidad)
# Muestra la tabla de trazabilidad

if len(trazabilidad) != 2:
    raise ValueError("Se esperaban exactamente dos series: compra y venta.")
# Comprueba que se hayan recibido exactamente las dos series esperadas

,codigo,nombre,unidad
0,PD04637PD,Tipo de cambio - TC Interbancario (S/ por US$)...,
1,PD04638PD,Tipo de cambio - TC Interbancario (S/ por US$)...,


### Celda 5 — Construcción del DataFrame y control de calidad

In [ ]:
def convertir_valor(valor):
    # Define una función para convertir cada valor recibido del BCRP a número

    if valor in (None, "", "n.d.", "n.d", "ND"):
        # Identifica valores vacíos o marcados como no disponibles
        return np.nan
        # Los convierte en NaN para que pandas los reconozca como datos faltantes

    return pd.to_numeric(str(valor).replace(",", ""), errors="coerce")
    # Convierte el valor a número y, si no puede hacerlo, lo convierte en NaN


filas = []
# Crea una lista vacía donde se almacenarán las filas de datos


for periodo in datos_json["periods"]:
    # Recorre todos los periodos recibidos desde el BCRP

    valores = periodo.get("values", [])
    # Obtiene los valores de las series correspondientes a ese periodo

    if len(valores) != len(config_series):
        # Comprueba que el periodo tenga la misma cantidad de valores que de series
        continue
        # Si no coincide, omite ese periodo

    fila = {"periodo": periodo.get("name")}
    # Crea una fila y guarda el nombre del periodo

    for i, serie in enumerate(config_series):
        # Recorre las series recibidas junto con su posición

        nombre = serie.get("name", f"serie_{i+1}")
        # Obtiene el nombre de la serie

        fila[nombre] = convertir_valor(valores[i])
        # Convierte el valor correspondiente y lo guarda en la fila

    filas.append(fila)
    # Agrega la fila completa a la lista


tc = pd.DataFrame(filas)
# Convierte la lista de filas en un DataFrame llamado tc

In [ ]:
MESES_ES = {
    "Ene": "01", "Feb": "02", "Mar": "03", "Abr": "04",
    "May": "05", "Jun": "06", "Jul": "07", "Ago": "08",
    "Set": "09", "Sep": "09", "Oct": "10", "Nov": "11", "Dic": "12",
}
# Crea una equivalencia entre las abreviaturas de meses en español y su número


def fecha_bcrp(texto):
    # Define una función para convertir las fechas del BCRP al formato de fecha de pandas

    partes = str(texto).strip().replace("-", ".").split(".")
    # Limpia el texto, reemplaza "-" por "." y separa día, mes y año

    if len(partes) != 3:
        # Comprueba que la fecha tenga tres partes
        return pd.NaT
        # Si no tiene el formato esperado, la marca como fecha no disponible

    dia, mes, anio = partes
    # Guarda por separado el día, mes y año

    mes = MESES_ES.get(mes.title(), mes)
    # Convierte el nombre del mes a su número correspondiente

    anio = f"20{anio}" if len(anio) == 2 else anio
    # Si el año tiene dos dígitos, agrega "20" delante

    return pd.to_datetime(
        f"{anio}-{mes}-{dia}",
        format="%Y-%m-%d",
        errors="coerce"
    )
    # Construye la fecha y la convierte al formato de fecha de pandas

In [ ]:
tc["fecha"] = tc["periodo"].map(fecha_bcrp)
# Aplica la función anterior a todos los periodos y crea la columna fecha

tc = tc.drop(columns="periodo").sort_values("fecha").drop_duplicates("fecha")
# Elimina la columna original, ordena por fecha y conserva una sola fila por fecha

columnas_tc = [c for c in tc.columns if c != "fecha"]
# Obtiene las columnas de tipo de cambio, excluyendo la fecha

In [ ]:
control = pd.DataFrame({
    "tipo": tc[columnas_tc].dtypes.astype(str),
    # Muestra el tipo de dato de cada serie

    "valores_validos": tc[columnas_tc].notna().sum(),
    # Cuenta cuántos valores disponibles tiene cada serie

    "valores_no_disponibles": tc[columnas_tc].isna().sum(),
    # Cuenta cuántos valores faltantes tiene cada serie

    "minimo": tc[columnas_tc].min(),
    # Obtiene el menor tipo de cambio registrado

    "maximo": tc[columnas_tc].max(),
    # Obtiene el mayor tipo de cambio registrado
})
# Construye una tabla para revisar la calidad de las series


print("Cobertura:", tc["fecha"].min().date(), "a", tc["fecha"].max().date())
# Muestra la primera y última fecha disponible


display(control)
# Muestra el resumen del control de calidad

display(tc.tail())
# Muestra las últimas filas del DataFrame

Cobertura: 2021-09-20 a 2026-09-17


,tipo,valores_validos,valores_no_disponibles,minimo,maximo
Tipo de cambio - TC Interbancario (S/ por US$) - Compra,float64,1240,64,3.340857,4.134833
Tipo de cambio - TC Interbancario (S/ por US$) - Venta,float64,1240,64,3.342000,4.137500


,Tipo de cambio - TC Interbancario (S/ por US$) - Compra,Tipo de cambio - TC Interbancario (S/ por US$) - Venta,fecha
1299,3.365571,3.367571,2026-09-11
1300,3.379000,3.380857,2026-09-14
1301,3.374571,3.376000,2026-09-15
1302,NaN,NaN,2026-09-16
1303,NaN,NaN,2026-09-17


### Pausa guiada 1

**1. ¿Qué evidencia confirma que los datos proceden del BCRP?**  
Se puede comprobar el origen porque la información fue consultada desde la **API pública de Series Estadísticas del BCRP** mediante los códigos oficiales **PD04637PD y PD04638PD**. Además, la solicitud respondió con **HTTP 200** y entregó las dos series previstas.

**2. ¿Por qué no debemos rellenar automáticamente los días con `n.d.`?**  
Porque `n.d.` indica que **no existe un dato disponible para ese día**. Sustituirlo de forma automática implicaría crear valores que no fueron registrados por el BCRP y podría modificar artificialmente los resultados del análisis.

**3. ¿Qué diferencia económica existe entre la serie compra y la serie venta?**  
El tipo de cambio de **compra** corresponde al valor al que se adquieren dólares del cliente, mientras que el de **venta** es el precio al que se ofrecen dólares. Como la empresa necesita adquirir dólares para cumplir su obligación, la referencia adecuada es la serie de **venta**.


## BLOQUE 2 — Retornos y volatilidad

### Celda 6 — Selección de la serie de venta por su nombre

In [ ]:
# Se buscan las columnas cuyo nombre contiene "venta"
candidatas_venta = [c for c in columnas_tc if "venta" in c.lower()]

# Verifica que exista exactamente una serie de venta
if len(candidatas_venta) != 1:
    raise ValueError(f"No se pudo identificar una única serie de venta: {candidatas_venta}")

# Guarda el nombre de la serie de venta encontrada
col_venta = candidatas_venta[0]

# Se buscan las columnas cuyo nombre contiene "compra"
candidatas_compra = [c for c in columnas_tc if "compra" in c.lower()]

# Guarda el nombre de compra si existe una única coincidencia
col_compra = candidatas_compra[0] if len(candidatas_compra) == 1 else None

# Selecciona fecha, venta y compra si fue encontrada
mercado = tc[["fecha", col_venta] + ([col_compra] if col_compra else [])].copy()

# Cambia los nombres a otros más simples para trabajar con ellos
mercado = mercado.rename(
    columns={
        col_venta: "tc_venta",
        **({col_compra: "tc_compra"} if col_compra else {})
    }
)

# Elimina filas que no tengan fecha o tipo de cambio de venta
mercado = mercado.dropna(subset=["fecha", "tc_venta"])

# Si existe la serie de compra, verifica que no sea mayor que la de venta
if "tc_compra" in mercado:
    inconsistencias = (mercado["tc_compra"] > mercado["tc_venta"]).sum()
    print("Filas con compra mayor que venta:", inconsistencias)

# Muestra qué serie fue seleccionada como principal
print("Serie principal:", col_venta)

# Muestra las últimas filas del DataFrame
display(mercado.tail())

Filas con compra mayor que venta: 1
Serie principal: Tipo de cambio - TC Interbancario (S/ por US$) - Venta


,fecha,tc_venta,tc_compra
1297,2026-09-09,3.358143,3.356429
1298,2026-09-10,3.371571,3.369857
1299,2026-09-11,3.367571,3.365571
1300,2026-09-14,3.380857,3.379000
1301,2026-09-15,3.376000,3.374571


### Celda 7 — Retornos logarítmicos y volatilidad

In [ ]:
mercado["retorno_log"] = np.log(
    mercado["tc_venta"] / mercado["tc_venta"].shift(1)
)
# Calcula el retorno logarítmico entre un día y el día anterior

retornos = mercado["retorno_log"].dropna()
# Elimina el primer valor, que queda vacío porque no tiene un día anterior

vol_diaria = retornos.tail(VENTANA_BASE).std(ddof=1)
# Calcula la desviación estándar de los últimos 252 retornos como medida de volatilidad diaria

vol_anualizada = vol_diaria * np.sqrt(252)
# Convierte la volatilidad diaria en una referencia anualizada usando 252 días

tc_actual = mercado["tc_venta"].iloc[-1]
# Obtiene el último tipo de cambio de venta disponible

print(f"Tipo de cambio de venta actual: S/ {tc_actual:.4f} por US$")
# Muestra el tipo de cambio actual con cuatro decimales

print(f"Volatilidad diaria ({VENTANA_BASE} observaciones): {vol_diaria:.4%}")
# Muestra la volatilidad diaria calculada con las últimas 252 observaciones

print(f"Volatilidad anualizada referencial: {vol_anualizada:.2%}")
# Muestra la volatilidad diaria expresada como referencia anual

fig = px.line(
    mercado,
    x="fecha",
    y="tc_venta",
    title="Tipo de cambio interbancario venta"
)
# Crea un gráfico de línea para observar la evolución del tipo de cambio de venta

fig.update_yaxes(title="S/ por US$")
# Coloca la unidad del eje vertical

fig.show()
# Muestra el gráfico

Tipo de cambio de venta actual: S/ 3.3760 por US$
Volatilidad diaria (252 observaciones): 0.4207%
Volatilidad anualizada referencial: 6.68%


### Pausa guiada 2

¿Qué significa para una empresa importadora que aumente la volatilidad aunque el tipo de cambio promedio permanezca similar?

**Respuesta:** Para una empresa importadora, una volatilidad mayor implica que el tipo de cambio puede presentar variaciones más amplias aun cuando su promedio se mantenga cercano. Esto incrementa la incertidumbre sobre cuánto deberá pagar en soles por los **USD 100,000**, por lo que la decisión debe considerar la distribución de posibles escenarios y no únicamente el promedio histórico.


## BLOQUE 3 — Simulación Monte Carlo

### Celda 8 — Bootstrap histórico de retornos

In [ ]:
def dias_habiles_aproximados(dias_calendario):
    return max(1, round(dias_calendario * 252 / 365))

def simular_tc_final(retornos_historicos, tc_inicial, horizonte_calendario,
                     escenarios=20_000, semilla=2026):
    horizonte_habil = dias_habiles_aproximados(horizonte_calendario)
    muestra = np.asarray(retornos_historicos.dropna(), dtype=float)
    if len(muestra) < 30:
        raise ValueError("No hay suficientes retornos históricos para simular.")
    rng = np.random.default_rng(semilla)
    caminos = rng.choice(muestra, size=(escenarios, horizonte_habil), replace=True)
    retornos_acumulados = caminos.sum(axis=1)
    tc_final = tc_inicial * np.exp(retornos_acumulados)
    return tc_final, horizonte_habil

retornos_base = retornos.tail(VENTANA_BASE)
tc_simulado, HORIZONTE_HABIL = simular_tc_final(
    retornos_base, tc_actual, HORIZONTE_CALENDARIO, ESCENARIOS, SEMILLA
)

resumen_tc = pd.Series(tc_simulado).describe(percentiles=[0.05, 0.50, 0.95])
print("Horizonte hábil aproximado:", HORIZONTE_HABIL)
display(resumen_tc.to_frame("tc_final_simulado"))

fig = px.histogram(
    x=tc_simulado, nbins=80,
    title="Distribución simulada del tipo de cambio al horizonte"
)
fig.add_vline(x=tc_actual, line_dash="dash", line_color="red", annotation_text="TC actual")
fig.update_xaxes(title="S/ por US$")
fig.update_yaxes(title="Frecuencia")
fig.show()

Horizonte hábil aproximado: 41


,tc_final_simulado
count,20000.000000
mean,3.350907
std,0.089763
min,2.951701
5%,3.198630
50%,3.352919
95%,3.494499
max,3.685291


### Pausa guiada 3

La distribución anterior no es un intervalo de confianza del tipo de cambio. Representa escenarios futuros simulados bajo el supuesto de que los retornos recientes son una referencia útil para el horizonte analizado.

## BLOQUE 4 — Comparación y motor de decisión

### Celda 9 — Comprar hoy frente a esperar

In [ ]:
costo_comprar_hoy = MONTO_USD * tc_actual
# Calcula cuánto costaría comprar los USD 100,000 al tipo de cambio actual

costos_esperar = MONTO_USD * tc_simulado
# Calcula el costo en soles para cada uno de los 20,000 escenarios simulados

costo_esperado = costos_esperar.mean()
# Calcula el costo promedio de esperar

prob_sobrecosto = np.mean(costos_esperar > costo_comprar_hoy)
# Calcula la probabilidad de que esperar resulte más caro que comprar hoy

p95_costo = np.percentile(costos_esperar, 95)
# Obtiene el costo que marca el percentil 95 de los escenarios

sobrecosto_esperado_pct = costo_esperado / costo_comprar_hoy - 1
# Calcula cuánto cambia el costo esperado respecto a comprar hoy

sobrecosto_p95_pct = p95_costo / costo_comprar_hoy - 1
# Calcula cuánto mayor sería el costo del P95 respecto a comprar hoy

comparacion = pd.DataFrame({
    "indicador": [
        "Costo de comprar hoy", "Costo esperado si se espera",
        "Probabilidad de sobrecosto", "Percentil 95 del costo",
        "Sobrecosto esperado (%)", "Sobrecosto P95 (%)"
    ],
    "valor": [
        costo_comprar_hoy, costo_esperado, prob_sobrecosto,
        p95_costo, sobrecosto_esperado_pct, sobrecosto_p95_pct
    ]
})
# Construye una tabla con los indicadores para comparar ambas alternativas

display(comparacion)
# Muestra los resultados

,indicador,valor
0,Costo de comprar hoy,337600.000000
1,Costo esperado si se espera,335090.712014
2,Probabilidad de sobrecosto,0.395050
3,Percentil 95 del costo,349449.919665
4,Sobrecosto esperado (%),-0.007433
5,Sobrecosto P95 (%),0.035100


### Celda 10 — Tabla de riesgo e interpretación

In [ ]:
tabla_riesgo = pd.DataFrame({
    "Métrica": [
        "TC actual", "TC esperado", "TC P95",
        "Costo hoy", "Costo esperado", "Costo P95",
        "Prob. sobrecosto"
    ],
    "Resultado": [
        tc_actual,
        tc_simulado.mean(),
        np.percentile(tc_simulado, 95),
        costo_comprar_hoy,
        costo_esperado,
        p95_costo,
        prob_sobrecosto
    ]
})
# Crea una tabla que resume los principales indicadores de tipo de cambio, costo y riesgo

display(tabla_riesgo)
# Muestra la tabla de resultados

print(
    f"Si la empresa espera, existe una probabilidad de {prob_sobrecosto:.1%} "
    f"de pagar más que si compra hoy. En el percentil 95, el costo alcanzaría "
    f"S/ {p95_costo:,.2f}."
)
# Muestra una interpretación automática de la probabilidad de sobrecosto y del escenario P95

,Métrica,Resultado
0,TC actual,3.376000
1,TC esperado,3.350907
2,TC P95,3.494499
3,Costo hoy,337600.000000
4,Costo esperado,335090.712014
5,Costo P95,349449.919665
6,Prob. sobrecosto,0.395050


Si la empresa espera, existe una probabilidad de 39.5% de pagar más que si compra hoy. En el percentil 95, el costo alcanzaría S/ 349,449.92.


### Celda 11 — Motor transparente de decisión

In [ ]:
def motor_decision(prob_sobrecosto, sobrecosto_p95_pct,
                   tolerancia_prob=0.40, tolerancia_p95=0.02):
    # Define una función que determina la decisión según los límites de riesgo

    razones = []
    # Crea una lista vacía para guardar las razones de la decisión

    if prob_sobrecosto > tolerancia_prob:
        # Verifica si la probabilidad de sobrecosto supera el límite permitido

        razones.append(
            f"probabilidad de sobrecosto {prob_sobrecosto:.1%} > límite {tolerancia_prob:.1%}"
        )
        # Guarda la razón si la probabilidad supera el límite

    if sobrecosto_p95_pct > tolerancia_p95:
        # Verifica si el sobrecosto P95 supera el límite permitido

        razones.append(
            f"sobrecosto P95 {sobrecosto_p95_pct:.1%} > límite {tolerancia_p95:.1%}"
        )
        # Guarda la razón si el sobrecosto P95 supera el límite

    decision = "COMPRAR HOY" if razones else "ESPERAR"
    # Si existe al menos una razón de riesgo, indica comprar hoy; si no, indica esperar

    return decision, razones or ["los dos indicadores permanecen dentro de los umbrales definidos"]
    # Devuelve la decisión y las razones encontradas

decision_base, razones_base = motor_decision(
    prob_sobrecosto,
    sobrecosto_p95_pct,
    TOLERANCIA_PROB_SOBRECOSTO,
    TOLERANCIA_SOBRECOSTO_P95,
)
# Ejecuta el motor usando los indicadores calculados y las tolerancias definidas

print("DECISIÓN DEL MOTOR:", decision_base)
# Muestra la decisión obtenida

print("Razones:")
# Muestra el encabezado de las razones

for razon in razones_base:
    print("-", razon)
# Recorre y muestra cada razón de la decisión

print("\nImportante: el motor aplica una política académica explícita; no reemplaza el juicio gerencial.")
# Aclara que la decisión depende de los umbrales definidos y no sustituye al criterio de la empresa

DECISIÓN DEL MOTOR: COMPRAR HOY
Razones:
- sobrecosto P95 3.5% > límite 2.0%

Importante: el motor aplica una política académica explícita; no reemplaza el juicio gerencial.


## BLOQUE 5 — Sensibilidad y robustez

### Celda 12 — Comparación de ventanas históricas

In [ ]:
def evaluar_ventana(n):
    # Define una función para evaluar una ventana histórica determinada

    muestra = retornos.tail(n)
    # Toma los últimos n retornos históricos

    simulados, _ = simular_tc_final(
        muestra, tc_actual, HORIZONTE_CALENDARIO, ESCENARIOS, SEMILLA
    )
    # Simula el tipo de cambio futuro usando esa ventana histórica

    costos = MONTO_USD * simulados
    # Convierte los tipos de cambio simulados en costos para los USD 100,000

    prob = np.mean(costos > costo_comprar_hoy)
    # Calcula la probabilidad de que esperar cueste más que comprar hoy

    p95 = np.percentile(costos, 95)
    # Calcula el costo correspondiente al percentil 95

    p95_pct = p95 / costo_comprar_hoy - 1
    # Calcula el sobrecosto del P95 respecto al costo de comprar hoy

    decision, _ = motor_decision(
        prob, p95_pct,
        TOLERANCIA_PROB_SOBRECOSTO,
        TOLERANCIA_SOBRECOSTO_P95,
    )
    # Obtiene la decisión aplicando los mismos límites de riesgo

    return {
        "ventana": n,
        "volatilidad_diaria": muestra.std(ddof=1),
        "costo_esperado": costos.mean(),
        "prob_sobrecosto": prob,
        "costo_p95": p95,
        "sobrecosto_p95_pct": p95_pct,
        "decision": decision,
    }
    # Devuelve los principales resultados de la ventana evaluada

sensibilidad = pd.DataFrame([evaluar_ventana(252), evaluar_ventana(504)])
# Compara una ventana de 252 días con otra de 504 días

display(sensibilidad)
# Muestra la tabla de sensibilidad

if sensibilidad["decision"].nunique() > 1:
    print("La decisión cambia: el resultado es sensible a la ventana histórica.")
    # Indica que la decisión cambia entre las ventanas
else:
    print("La decisión no cambia entre ventanas, pero siguen existiendo supuestos no modelados.")
    # Indica que la decisión se mantiene, aunque existen otros factores no considerados

,ventana,volatilidad_diaria,costo_esperado,prob_sobrecosto,costo_p95,sobrecosto_p95_pct,decision
0,252,0.004207,335090.712014,0.39505,349449.919665,0.035100,COMPRAR HOY
1,504,0.003648,334875.226324,0.36530,347327.841541,0.028815,COMPRAR HOY


La decisión no cambia entre ventanas, pero siguen existiendo supuestos no modelados.


## Reto de aplicación en clase

La gerencia reduce su tolerancia máxima de probabilidad de sobrecosto a **25 %**. Modifique únicamente ese umbral, vuelva a ejecutar el motor y compare la decisión con la original.

In [ ]:
TOLERANCIA_RETO = 0.25
# Cambia únicamente el límite máximo de probabilidad de sobrecosto a 25%

decision_reto, razones_reto = motor_decision(
    prob_sobrecosto,
    sobrecosto_p95_pct,
    tolerancia_prob=TOLERANCIA_RETO,
    tolerancia_p95=TOLERANCIA_SOBRECOSTO_P95,
)
# Ejecuta nuevamente el motor usando el nuevo límite de 25%

print("Decisión original:", decision_base)
# Muestra la decisión obtenida con el límite original de 40%

print("Decisión con tolerancia de 25 %:", decision_reto)
# Muestra la decisión utilizando el nuevo límite

print("Razones del nuevo resultado:")
# Muestra el encabezado de las razones

for razon in razones_reto:
    print("-", razon)
# Muestra las condiciones que provocaron la nueva decisión

Decisión original: COMPRAR HOY
Decisión con tolerancia de 25 %: COMPRAR HOY
Razones del nuevo resultado:
- probabilidad de sobrecosto 39.5% > límite 25.0%
- sobrecosto P95 3.5% > límite 2.0%


## Síntesis ejecutiva guiada

Redacte durante la clase una síntesis breve que incluya:

- **Evidencia:** TC actual, costo hoy, costo esperado, probabilidad de sobrecosto y P95.
- **Inferencia:** qué revela la distribución simulada.
- **Decisión:** comprar hoy o esperar y por qué.
- **Condición de cambio:** qué umbral o supuesto haría cambiar la decisión.
- **Limitación:** al menos dos elementos no modelados, como spread, costo de financiamiento o instrumentos de cobertura.

## Síntesis ejecutiva

Los escenarios simulados toman como referencia un tipo de cambio actual de **S/ 3.3760 por US$**.

Si la empresa adquiere hoy los **US$ 100,000**, tendría que desembolsar **S/ 337,600.00**. En la simulación del horizonte futuro, el costo promedio estimado resulta en **S/ 335,090.71**, aunque existen escenarios en los que esperar genera un desembolso superior.

La probabilidad estimada de que esperar sea más costoso que comprar ahora es de **39.5%**. Asimismo, el percentil 95 alcanza **S/ 349,449.92**, mostrando un nivel de costo posible dentro de los escenarios más desfavorables simulados.

Con estos resultados, el motor mantiene la decisión de **COMPRAR HOY**, debido a que el P95 implica un sobrecosto aproximado de **3.5%**, por encima del límite aceptado de **2%**. Cuando la tolerancia máxima de probabilidad se reduce de 40% a 25%, la recomendación tampoco cambia.

La decisión podría modificarse si se ajustaran los umbrales de riesgo o los supuestos utilizados en la simulación. Además, el modelo no incorpora elementos como el **spread entre compra y venta**, los **costos de financiamiento** ni posibles **instrumentos de cobertura cambiaria**. Por ello, los resultados deben utilizarse como apoyo para la evaluación del riesgo y no como una predicción definitiva.


## Ticket de salida

En una frase, responda la pregunta orientadora e indique una evidencia concreta producida por el notebook.

**Respuesta:**  

La decisión frente al riesgo cambiario puede sustentarse comparando los costos de distintos escenarios simulados; en este caso, el notebook obtuvo una **probabilidad de sobrecosto de 39.5%** y un **P95 de S/ 349,449.92** para una obligación de **USD 100,000**.

---
